# Three-Model Hierarchical CNN Experiment

Plain CNN only. No multibranch classifier params.

Hierarchy:

1. `is_target`: Target vs Non-Target
2. `orientation`: trained on Target sequences only
3. `gesture_action`: trained on Target sequences only

Final compressed hierarchy label:

- Target rows: reconstructed original `gesture` using `(orientation, gesture_action)` lookup
- Non-Target rows: `Non-Target`

This is designed to test whether the hierarchy improves the target-gesture path before adding a separate non-target classifier.

In [1]:
from __future__ import annotations

import os
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report, confusion_matrix

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
except Exception as exc:
    BayesSearchCV = None
    Categorical = None
    Integer = None
    Real = None
    print("BayesSearchCV unavailable. Use search_mode='grid'.", exc)

import data_utils
import utils_hierarchical_cnn_aug as utils

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [2]:
# ============================================================
# Config
# ============================================================

search_mode = "grid"      # "grid" or "bayesian"
random_state = 42
holdout_size = 0.2
n_cv_splits = 3
n_iter_bayes = 12
n_jobs = 1

pipe_name = "sequence_builder"
classifier_name = "classifier"
corrector_name = "orientation_corrector"

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
print("timestamp:", timestamp)

timestamp: 20260515_0158


In [3]:
# ============================================================
# Load data
# ============================================================

data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

print("raw_train_df:", raw_train_df.shape)
print("train_demo_df:", train_demo_df.shape)
print("train sequences:", raw_train_df["sequence_id"].nunique())
print("subjects:", raw_train_df["subject"].nunique())

Using local data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data
raw_train_df: (574945, 341)
train_demo_df: (81, 8)
train sequences: 8151
subjects: 81


In [4]:
# ============================================================
# Base dataframe + helper targets
# ============================================================

train_df = raw_train_df.set_index("row_id").copy(deep=True)

train_df.loc[:, "gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df.loc[:, "gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df.loc[:, "is_target"] = train_df["sequence_type"].eq("Target").astype(int)

train_df.loc[:, "hierarchical_gesture"] = np.where(
    train_df["sequence_type"].eq("Target"),
    train_df["gesture"],
    "Non-Target",
)

train_df.loc[:, "orientation_action"] = (
    train_df["orientation"].astype(str)
    + "||"
    + train_df["gesture_action"].astype(str)
)

seq_counts = (
    train_df
    .drop_duplicates("sequence_id")
    ["hierarchical_gesture"]
    .value_counts()
)

display(seq_counts.head(30))
print("hierarchical classes:", train_df["hierarchical_gesture"].nunique())
print("gesture_action classes:", train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_action"].nunique())
print("orientation classes:", train_df.loc[train_df["sequence_type"].eq("Target"), "orientation"].nunique())

hierarchical_gesture
Non-Target                  3038
Eyelash - pull hair          640
Forehead - pull hairline     640
Neck - scratch               640
Neck - pinch skin            640
Forehead - scratch           640
Eyebrow - pull hair          638
Above ear - pull hair        638
Cheek - pinch skin           637
Name: count, dtype: int64

hierarchical classes: 9
gesture_action classes: 4
orientation classes: 4


In [5]:
# ============================================================
# Mapping check: orientation + gesture_action -> original gesture
# ============================================================

target_map_df = (
    train_df
    .loc[train_df["sequence_type"].eq("Target")]
    .drop_duplicates(["orientation", "gesture_action", "gesture"])
    [["orientation", "gesture_action", "gesture"]]
    .copy()
)

mapping_check = (
    target_map_df
    .groupby(["orientation", "gesture_action"])["gesture"]
    .nunique()
    .reset_index(name="n_gesture_labels")
)

display(mapping_check.sort_values("n_gesture_labels", ascending=False).head(20))
print("all mappings unique:", mapping_check["n_gesture_labels"].eq(1).all())

target_lookup = {
    (row.orientation, row.gesture_action): row.gesture
    for row in target_map_df.itertuples(index=False)
}

most_common_target_gesture = (
    train_df
    .loc[train_df["sequence_type"].eq("Target")]
    .drop_duplicates("sequence_id")
    ["gesture"]
    .mode()
    .iloc[0]
)

print("target lookup size:", len(target_lookup))
print("fallback target gesture:", most_common_target_gesture)

,orientation,gesture_action,n_gesture_labels
1,Lie on Back,pull hair,3
5,Lie on Side - Non Dominant,pull hair,3
13,Seated Straight,pull hair,3
9,Seated Lean Non Dom - FACE DOWN,pull hair,3
7,Lie on Side - Non Dominant,scratch,2
4,Lie on Side - Non Dominant,pinch skin,2
3,Lie on Back,scratch,2
0,Lie on Back,pinch skin,2
15,Seated Straight,scratch,2
12,Seated Straight,pinch skin,2


all mappings unique: False
target lookup size: 16
fallback target gesture: Eyelash - pull hair


In [6]:
# ============================================================
# Subject holdout split
# ============================================================

seq_meta = (
    train_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "subject", "sequence_type", "gesture", "hierarchical_gesture"]]
    .reset_index(drop=True)
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=holdout_size,
    random_state=random_state,
)

train_seq_idx, holdout_seq_idx = next(
    splitter.split(
        seq_meta,
        y=seq_meta["hierarchical_gesture"],
        groups=seq_meta["subject"],
    )
)

train_seq_ids = seq_meta.loc[train_seq_idx, "sequence_id"]
holdout_seq_ids = seq_meta.loc[holdout_seq_idx, "sequence_id"]

train_model_df = train_df.loc[train_df["sequence_id"].isin(train_seq_ids)].copy()
holdout_df = train_df.loc[train_df["sequence_id"].isin(holdout_seq_ids)].copy()

target_only_train_df = train_model_df.loc[train_model_df["sequence_type"].eq("Target")].copy()
target_only_holdout_df = holdout_df.loc[holdout_df["sequence_type"].eq("Target")].copy()

print("train sequences:", train_model_df["sequence_id"].nunique())
print("holdout sequences:", holdout_df["sequence_id"].nunique())
print("target-only train sequences:", target_only_train_df["sequence_id"].nunique())
print("target-only holdout sequences:", target_only_holdout_df["sequence_id"].nunique())
print("train subjects:", train_model_df["subject"].nunique())
print("holdout subjects:", holdout_df["subject"].nunique())
print("subject overlap:", len(set(train_model_df["subject"]) & set(holdout_df["subject"])))

train sequences: 6422
holdout sequences: 1729
target-only train sequences: 4025
target-only holdout sequences: 1088
train subjects: 64
holdout subjects: 17
subject overlap: 0


In [ ]:
# ============================================================
# Common plain-CNN param options
# No multibranch params here.
# ============================================================

if search_mode == "bayesian":
    assert BayesSearchCV is not None, "BayesSearchCV is not available. Set search_mode='grid'."

    target_param_space = {
        f"{pipe_name}__acc_modes": Categorical([
            ("raw",),
            ("raw", "velocity"),
            ("raw", "velocity", "displacement"),
        ]),
        f"{pipe_name}__linear_acc_mode": Categorical([None, "baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([False, True]),
        f"{pipe_name}__sampling_rate": Categorical([20, 25, 50]),
        f"{pipe_name}__clip_value": Categorical([None, 20.0, 50.0]),
        f"{pipe_name}__interp_mode": Categorical([None, "linear"]),
        f"{pipe_name}__window_size": Integer(3, 21),
        f"{pipe_name}__smooth_alpha": Categorical([None, 0.5, 0.8]),
        f"{pipe_name}__standardize": Categorical(["mean_std"]),
        f"{pipe_name}__rotation_modes": Categorical([
            ("quaternion",),
            ("rot6d",),
            ("quaternion", "rot6d"),
        ]),
        f"{pipe_name}__tof_mode": Categorical([None, "sensor_stats"]),
        f"{pipe_name}__tof_fill_mode": Categorical(["nan_interpolate", "far_255"]),
        f"{pipe_name}__thm_mode": Categorical([None, "centered"]),

        f"{classifier_name}__maxlen": Integer(32, 160),
        f"{classifier_name}__conv_filters": Categorical(["64", "128", "128-256", "128-256-256"]),
        f"{classifier_name}__kernel_sizes": Categorical(["3", "5", "3-3", "5-5", "5-5-5"]),
        f"{classifier_name}__pool_sizes": Categorical(["none", "2", "2-2", "2-2-2"]),
        f"{classifier_name}__use_batch_norm": Categorical([True]),
        f"{classifier_name}__spatial_dropout": Real(0.0, 0.5),
        f"{classifier_name}__dense_units": Categorical(["16", "32", "64", "128"]),
        f"{classifier_name}__dropout": Real(0.0, 0.6),
        f"{classifier_name}__learning_rate": Real(1e-6, 2e-3, prior="log-uniform"),
        f"{classifier_name}__batch_size": Categorical([32, 64]),
        f"{classifier_name}__epochs": Categorical([80, 120, 200]),
        f"{classifier_name}__patience": Categorical([10, 20]),

        f"{classifier_name}__use_mixup": Categorical([False, True]),
        f"{classifier_name}__mixup_alpha": Real(0.2, 0.6),
        f"{classifier_name}__mixup_size": Real(0.5, 1.5),
        f"{classifier_name}__use_gaussian_noise": Categorical([False, True]),
        f"{classifier_name}__noise_std": Real(0.001, 0.05, prior="log-uniform"),
        f"{classifier_name}__use_time_mask": Categorical([False, True]),
        f"{classifier_name}__time_mask_ratio": Real(0.05, 0.2),
        f"{classifier_name}__use_magnitude_scaling": Categorical([False, True]),
        f"{classifier_name}__use_modality_dropout": Categorical([False, True]),
        f"{classifier_name}__drop_tof_prob": Real(0.0, 0.4),
        f"{classifier_name}__drop_thm_prob": Real(0.0, 0.4),
    }

    orientation_param_space = target_param_space.copy()
    orientation_param_space.update({
        f"{pipe_name}__acc_modes": Categorical([
            ("displacement",),
            ("velocity", "displacement"),
            ("raw", "velocity", "displacement"),
        ]),
        f"{pipe_name}__rotation_modes": Categorical([
            ("rot6d",),
            ("quaternion", "rot6d"),
            ("euler", "rot6d"),
        ]),
    })

    action_param_space = target_param_space.copy()
    action_param_space.update({
        f"{pipe_name}__acc_modes": Categorical([
            ("raw", "jerk"),
            ("velocity", "jerk"),
            ("raw", "velocity", "jerk"),
            ("raw", "velocity", "displacement", "jerk"),
        ]),
        f"{pipe_name}__rotation_modes": Categorical([
            ("angular_velocity",),
            ("delta_euler", "angular_velocity"),
            ("quaternion", "rot6d", "angular_velocity"),
        ]),
    })

elif search_mode == "grid":
    target_param_space = {
        f"{pipe_name}__acc_modes": [
            # ("raw",),
            ('smoothed', "velocity", "displacement", "jerk"),
        ],
        f"{pipe_name}__linear_acc_mode": ["baseline"],
        f"{pipe_name}__use_acc_magnitude": [True],
        f"{pipe_name}__use_linear_acc_magnitude": [True],
        f"{pipe_name}__sampling_rate": [10],
        f"{pipe_name}__clip_value": [50.0],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__window_size": [30],
        f"{pipe_name}__smooth_alpha": [0.8],
        f"{pipe_name}__standardize": ["mean_std"],
        f"{pipe_name}__rotation_modes": [("quaternion","euler","rot6d", "angular_velocity")],
        f"{pipe_name}__tof_mode": ["sensor_stats"],
        f"{pipe_name}__tof_fill_mode": ["far_255"],
        f"{pipe_name}__thm_mode": ["centered"],

        f"{classifier_name}__maxlen": [170],
        f"{classifier_name}__conv_filters": ["128-256-256"],
        f"{classifier_name}__kernel_sizes": ["5-5-5"],
        f"{classifier_name}__pool_sizes": ["2-2-2"],
        f"{classifier_name}__use_batch_norm": [True],
        f"{classifier_name}__spatial_dropout": [0.35],
        f"{classifier_name}__dense_units": ["16"],
        f"{classifier_name}__dropout": [0.45],
        f"{classifier_name}__learning_rate": [5.7e-6],
        f"{classifier_name}__batch_size": [32],
        f"{classifier_name}__epochs": [200],
        f"{classifier_name}__patience": [20],

        f"{classifier_name}__use_mixup": [True],
        f"{classifier_name}__mixup_alpha": [0.4],
        f"{classifier_name}__mixup_size": [1.0],
        f"{classifier_name}__use_gaussian_noise": [False],
        f"{classifier_name}__use_time_mask": [False],
        f"{classifier_name}__use_magnitude_scaling": [False],
        f"{classifier_name}__use_modality_dropout": [False],
    }


else:
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

In [8]:
# ============================================================
# Target vs Non-Target model
# ============================================================

target_name = "is_target"

pipeline_target = Pipeline([
    (corrector_name, utils.SensorOrientationCorrector(demo_df=train_demo_df)),
    (pipe_name, utils.MultiDomainSequenceExtractor()),
    (classifier_name, utils.KerasAugmentedCNN1DSequenceClassifier(
        target=target_name,
        verbose=0,
        random_state=random_state,
    )),
])

cv_target = GroupKFold(n_splits=n_cv_splits)

y_target = train_model_df[["sequence_id", target_name]].copy()
groups_target = train_model_df["subject"].copy()

if search_mode == "bayesian":
    search_target = BayesSearchCV(
        estimator=pipeline_target,
        search_spaces=target_param_space,
        n_iter=n_iter_bayes,
        scoring=None,
        cv=cv_target,
        n_jobs=n_jobs,
        refit=True,
        random_state=random_state,
        verbose=2,
    )
else:
    search_target = GridSearchCV(
        estimator=pipeline_target,
        param_grid=target_param_space,
        scoring=None,
        cv=cv_target,
        n_jobs=n_jobs,
        refit=True,
        verbose=2,
    )

search_target.fit(train_model_df, y_target, groups=groups_target)

target_results_df = pd.DataFrame(search_target.cv_results_)
target_results_path = results_dir / f"cv_results_target_vs_non_target_{timestamp}.csv"
target_results_df.to_csv(target_results_path, index=False)

print("best target score:", search_target.best_score_)
print("best target params:")
display(pd.Series(search_target.best_params_))
print("saved:", target_results_path)

Fitting 3 folds for each of 64 candidates, totalling 192 fits


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Orientation model, Target only
# ============================================================

target_name = "orientation"

pipeline_orientation = Pipeline([
    (corrector_name, utils.SensorOrientationCorrector(demo_df=train_demo_df)),
    (pipe_name, utils.MultiDomainSequenceExtractor()),
    (classifier_name, utils.KerasAugmentedCNN1DSequenceClassifier(
        target=target_name,
        verbose=0,
        random_state=random_state,
    )),
])

cv_orientation = GroupKFold(n_splits=n_cv_splits)

y_orientation = target_only_train_df[["sequence_id", target_name]].copy()
groups_orientation = target_only_train_df["subject"].copy()

if search_mode == "bayesian":
    search_orientation = BayesSearchCV(
        estimator=pipeline_orientation,
        search_spaces=orientation_param_space,
        n_iter=n_iter_bayes,
        scoring=None,
        cv=cv_orientation,
        n_jobs=n_jobs,
        refit=True,
        random_state=random_state,
        verbose=2,
    )
else:
    search_orientation = GridSearchCV(
        estimator=pipeline_orientation,
        param_grid=orientation_param_space,
        scoring=None,
        cv=cv_orientation,
        n_jobs=n_jobs,
        refit=True,
        verbose=2,
    )

search_orientation.fit(target_only_train_df, y_orientation, groups=groups_orientation)

orientation_results_df = pd.DataFrame(search_orientation.cv_results_)
orientation_results_path = results_dir / f"cv_results_orientation_{timestamp}.csv"
orientation_results_df.to_csv(orientation_results_path, index=False)

print("best orientation score:", search_orientation.best_score_)
print("best orientation params:")
display(pd.Series(search_orientation.best_params_))
print("saved:", orientation_results_path)

In [ ]:
# ============================================================
# Gesture action model, Target only
# ============================================================

target_name = "gesture_action"

pipeline_action = Pipeline([
    (corrector_name, utils.SensorOrientationCorrector(demo_df=train_demo_df)),
    (pipe_name, utils.MultiDomainSequenceExtractor()),
    (classifier_name, utils.KerasAugmentedCNN1DSequenceClassifier(
        target=target_name,
        verbose=0,
        random_state=random_state,
    )),
])

cv_action = GroupKFold(n_splits=n_cv_splits)

y_action = target_only_train_df[["sequence_id", target_name]].copy()
groups_action = target_only_train_df["subject"].copy()

if search_mode == "bayesian":
    search_action = BayesSearchCV(
        estimator=pipeline_action,
        search_spaces=action_param_space,
        n_iter=n_iter_bayes,
        scoring=None,
        cv=cv_action,
        n_jobs=n_jobs,
        refit=True,
        random_state=random_state,
        verbose=2,
    )
else:
    search_action = GridSearchCV(
        estimator=pipeline_action,
        param_grid=action_param_space,
        scoring=None,
        cv=cv_action,
        n_jobs=n_jobs,
        refit=True,
        verbose=2,
    )

search_action.fit(target_only_train_df, y_action, groups=groups_action)

action_results_df = pd.DataFrame(search_action.cv_results_)
action_results_path = results_dir / f"cv_results_gesture_action_{timestamp}.csv"
action_results_df.to_csv(action_results_path, index=False)

print("best action score:", search_action.best_score_)
print("best action params:")
display(pd.Series(search_action.best_params_))
print("saved:", action_results_path)

In [ ]:
# ============================================================
# Holdout evaluation: combine three models
# ============================================================

holdout_seq = (
    holdout_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "subject", "sequence_type", "gesture", "hierarchical_gesture"]]
    .reset_index(drop=True)
)

is_target_pred = search_target.best_estimator_.predict(holdout_df)
orientation_pred = search_orientation.best_estimator_.predict(holdout_df)
action_pred = search_action.best_estimator_.predict(holdout_df)

holdout_seq.loc[:, "pred_is_target"] = is_target_pred.astype(int)
holdout_seq.loc[:, "pred_orientation"] = orientation_pred.astype(str)
holdout_seq.loc[:, "pred_gesture_action"] = action_pred.astype(str)

combo_keys = list(zip(
    holdout_seq["pred_orientation"].astype(str),
    holdout_seq["pred_gesture_action"].astype(str),
))

mapped_target_pred = pd.Series(combo_keys).map(target_lookup)
mapped_target_pred = mapped_target_pred.fillna(most_common_target_gesture)

holdout_seq.loc[:, "prediction"] = np.where(
    holdout_seq["pred_is_target"].eq(1),
    mapped_target_pred.to_numpy(),
    "Non-Target",
)

hier_f1 = f1_score(
    holdout_seq["hierarchical_gesture"],
    holdout_seq["prediction"],
    average="macro",
)

target_detector_f1 = f1_score(
    holdout_seq["sequence_type"].eq("Target").astype(int),
    holdout_seq["pred_is_target"].astype(int),
    average="macro",
)

print("Hierarchical holdout macro F1:", round(hier_f1, 4))
print("Target detector holdout macro F1:", round(target_detector_f1, 4))
print("Invalid target combos fallback count:", int(mapped_target_pred.isna().sum()))

print("
Target detector report")
print(classification_report(
    holdout_seq["sequence_type"].eq("Target").astype(int),
    holdout_seq["pred_is_target"].astype(int),
))

print("
Hierarchical report")
print(classification_report(
    holdout_seq["hierarchical_gesture"],
    holdout_seq["prediction"],
))

holdout_results_path = results_dir / f"holdout_results_three_model_hierarchy_cnn_{timestamp}.csv"
holdout_seq.to_csv(holdout_results_path, index=False)
print("saved:", holdout_results_path)

In [ ]:
# ============================================================
# Save compact summary
# ============================================================

summary_df = pd.DataFrame([
    {
        "timestamp": timestamp,
        "search_mode": search_mode,
        "holdout_size": holdout_size,
        "n_cv_splits": n_cv_splits,
        "target_cv_best_score": search_target.best_score_,
        "orientation_cv_best_score": search_orientation.best_score_,
        "gesture_action_cv_best_score": search_action.best_score_,
        "target_detector_holdout_macro_f1": target_detector_f1,
        "hierarchical_holdout_macro_f1": hier_f1,
        "train_sequences": train_model_df["sequence_id"].nunique(),
        "holdout_sequences": holdout_df["sequence_id"].nunique(),
        "train_subjects": train_model_df["subject"].nunique(),
        "holdout_subjects": holdout_df["subject"].nunique(),
        "target_best_params": json.dumps(search_target.best_params_, default=str),
        "orientation_best_params": json.dumps(search_orientation.best_params_, default=str),
        "action_best_params": json.dumps(search_action.best_params_, default=str),
    }
])

summary_path = results_dir / f"summary_three_model_hierarchy_cnn_{timestamp}.csv"
summary_df.to_csv(summary_path, index=False)
display(summary_df)
print("saved:", summary_path)